<p align="center">
  <img src="./assets/banners/03-hear-your-customers.svg" alt="MAI Models · Multimodal Product Launch — Hear your customers: How can MAI-Transcribe-2 turn launch audio into answers? Microsoft AI Samples" width="900" />
</p>

# 👂 Hear your customers

> **How can MAI-Transcribe-2 turn launch audio into answers we can act on — and how do we know the transcript is right?**

⏱️ About 10 minutes to review, or about 10 to run live · 🧱 You'll build: checked transcripts of the launch audio, and a way to search them

**By the end, you'll be able to:**

- Transcribe audio with MAI-Transcribe-2, and give it hints for unusual product names
- Score a transcript against what was really said, with word error rate
- Explain what transcription gets right, and what a person still checks

> 🧪 **Heads-up: you can read this one or run it.** In replay mode (the default) it shows saved transcripts from an earlier run, in `examples/03/`. To run it live, run notebook 02 live first: we transcribe its clips.

> 💡 **How to use this notebook:** run each cell in order with **Shift + Enter**. Before each code cell, read the note and **take a guess**.

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.1 The story: we wrote the answer key

The launch is live. Ads are on the radio, the hotline is ringing, and the support team is training on example calls. Contoso wants to know what's being said, and to search it.

Here's the trick that makes this notebook useful: in notebook 02 **we wrote every script**. So we know exactly what was said, and we can score the transcript word by word.

That's a habit worth keeping for any AI system: **measure against a fixed answer key, decided before we look at the results.**

### Pick your route

| Route | Sections | Good for |
| --- | --- | --- |
| ⚡ **Short** (about 5 minutes to review) | 3.4, 3.5, 3.6, 3.9 | A first look before a customer meeting |
| 🧭 **Full** (about 10 minutes to review) | All of them | Learning every feature well enough to teach it |

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.2 New words

| Word | In plain words |
| --- | --- |
| **Transcription** | Turning speech into text |
| **Word error rate (WER)** | The share of words we'd have to change to fix a transcript. 0 is perfect; 0.1 means one word in ten is wrong |
| **Keyword hints** | A list of names you expect to hear, so the model spells them your way. The service calls it a phrase list |
| **Diarization** | Working out who spoke when, in a recording with several people |
| **Streaming** | Transcribing while someone is still talking, for live captions |

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.3 Before you start

**Just reading?** Run the next cell, then go to 3.4: replay mode needs no Azure.

**Running it live?** You need the same setup as notebook 02, plus its audio:

| You need | Why |
| --- | --- |
| A Foundry resource in a region with MAI-Transcribe | MAI-Transcribe-2 comes with the resource; there's nothing to deploy. `eastus` works ([docs](https://learn.microsoft.com/azure/ai-services/speech-service/mai-transcribe)) |
| The **Foundry User** role, and a sign-in | Keyless access to speech. Run `az login` in a terminal |
| The clips from notebook 02 in `output/02/` | We transcribe them |
| Live mode | Set `DEMO_MODE = "live"` in the next cell, or add `DEMO_MODE=live` to this folder's `.env` |

**What it costs.** You pay per hour of audio; our clips add up to about a minute. See [Speech pricing](https://aka.ms/SpeechPricing).

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "contoso-retail").is_dir())
load_dotenv(ROOT / "workshops" / "mai-multimodal-product-launch" / ".env")

DEMO_MODE = None  # ✏️ Set to "live" or "replay" to override .env. Leave None to use .env.

mode = (DEMO_MODE or os.environ.get("DEMO_MODE") or "replay").strip().lower()
if mode not in ("replay", "live"):
    raise SystemExit(f'⚠️ DEMO_MODE is "{mode}". Use "replay" or "live".')
source = "this cell" if DEMO_MODE else ".env" if os.environ.get("DEMO_MODE") else "the default"
print(f"🎬 Mode: {'🔁 replay' if mode == 'replay' else '👂 live'} (from {source})")

🎬 Mode: 👂 live (from the repo's .env)


<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.4 Get set up

Two cells: settings, then our helpers. `transcribe()` sends a clip and saves the reply; `word_error_rate()` compares it with the script.

> 🔮 **Guess first:** our scoring ignores capital letters and punctuation. Is that fair, or does it hide real mistakes?

In [ ]:
# Settings come from this folder's .env, which infra/setup.sh writes, so no names or secrets are written into the code.
import difflib
import json
import os
import re
import time
import unicodedata
from pathlib import Path
from urllib.parse import urlparse

import requests
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "contoso-retail").is_dir())
HERE = ROOT / "workshops" / "mai-multimodal-product-launch"
load_dotenv(HERE / ".env")


def setting(name: str, default: str | None = None) -> str:
    """Return a setting from .env, or stop with a message that says how to fix it."""
    value = os.environ.get(name, "").strip()
    if value and "<" not in value:
        return value
    if default is not None:
        return default
    raise SystemExit(f"⚠️ {name} isn't set yet. Run `bash infra/setup.sh` (it writes .env), then run this cell again.")


def clean(text: str | None) -> str:
    """Keep error messages short, and free of IDs and addresses, before they land in saved output."""
    text = re.sub(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}", "<id>", text or "")
    return re.sub(r"(?:https?|wss?)://\S+", "<address>", text)[:240]


# The mode cell in 3.3 wins over .env. Replay shows saved results and needs no Azure at all.
REPLAY = (globals().get("DEMO_MODE") or setting("DEMO_MODE", "replay")).strip().lower() == "replay"
OUTPUT = HERE / "output" / "03"      # Your results. Git ignores this folder.
EXAMPLES = HERE / "examples" / "03"  # Results from one of our runs, so anyone can replay the notebook
OUTPUT.mkdir(parents=True, exist_ok=True)

TRANSCRIBE = setting("MODEL_TRANSCRIBE", "MAI-Transcribe-2")
API_VERSION = "2025-10-15"
VOICE_OUTPUT, VOICE_EXAMPLES = HERE / "output" / "02", HERE / "examples" / "02"

credential = DefaultAzureCredential()  # Keyless: uses your az login, not a secret key.

print(f"✅ Settings loaded. Transcription model: {TRANSCRIBE}")
print(f"   Mode: {'🔁 replay (your transcripts first, then examples/03/)' if REPLAY else '👂 live'}")

✅ Settings loaded. Transcription model: MAI-Transcribe-2
   Mode: 👂 live


Here are the helpers. Notice `words()`: it writes down our scoring rules *before* we see any scores. A frozen measuring stick, in miniature.

In [ ]:
HINTS = {
    401: "Sign-in failed. Run `az login` in a terminal, then run this cell again.",
    403: "You're signed in, but don't have access yet. You need the Foundry User role on the Foundry resource.",
    404: "That address or model wasn't found. Check that your resource is in a region with MAI-Transcribe.",
}


def audio(name: str) -> Path | None:
    """Find a clip from notebook 02: yours in output/02/ first, then ours in examples/02/."""
    for folder in (VOICE_OUTPUT, VOICE_EXAMPLES):
        if (folder / f"{name}.mp3").exists():
            return folder / f"{name}.mp3"
    return None


def scripts() -> dict:
    """What notebook 02 told the voice to say. These are our answer key."""
    for folder in (VOICE_OUTPUT, VOICE_EXAMPLES):
        if (folder / "scripts.json").exists():
            return json.loads((folder / "scripts.json").read_text())
    return {}


def transcribe(name: str, clip: str, hints: list[str] | None = None, speakers: int | None = None) -> dict | None:
    """Send one clip to MAI-Transcribe-2 and save the reply. Returns it, or None if there's nothing to work with."""
    for path in (OUTPUT / f"{name}.json", EXAMPLES / f"{name}.json"):
        if REPLAY and path.exists():
            saved = json.loads(path.read_text())
            print(f"🔁 {name} · {saved['model']} · {saved['seconds']} s ({'saved earlier' if path.parent == OUTPUT else 'from examples/03/'})")
            return saved
    if REPLAY:
        print(f"⏭️ {name}: no saved transcript yet. Set DEMO_MODE=live to make it.")
        return None
    source = audio(clip)
    if not source:
        print(f"⏭️ {name}: no audio for {clip}. Run notebook 02 first.")
        return None
    definition: dict = {"enhancedMode": {"enabled": True, "model": TRANSCRIBE}}
    if hints:
        definition["phraseList"] = {"phrases": hints}
    if speakers:
        definition["diarization"] = {"enabled": True}  # MAI-Transcribe-2 rejects maxSpeakers for now
    resource = urlparse(setting("FOUNDRY_PROJECT_ENDPOINT")).hostname.split(".")[0]
    token = credential.get_token("https://cognitiveservices.azure.com/.default").token
    started = time.perf_counter()
    reply = requests.post(f"https://{resource}.cognitiveservices.azure.com/speechtotext/transcriptions:transcribe?api-version={API_VERSION}",
                          headers={"Authorization": f"Bearer {token}"}, timeout=300,
                          files={"audio": (source.name, source.read_bytes(), "audio/mpeg"),
                                 "definition": (None, json.dumps(definition), "application/json")})
    seconds = round(time.perf_counter() - started, 2)
    if not reply.ok:
        try:
            detail = reply.json().get("error", {}).get("message", "")
        except ValueError:
            detail = ""
        raise SystemExit(f"⚠️ {reply.status_code} from {TRANSCRIBE}. {HINTS.get(reply.status_code, 'The message below usually says what to change.')}\n{clean(detail)}")
    result = {"model": TRANSCRIBE, "seconds": seconds, "clip": clip, "definition": definition, "reply": reply.json()}
    (OUTPUT / f"{name}.json").write_text(json.dumps(result, indent=2, ensure_ascii=False))
    print(f"👂 {name} · {TRANSCRIBE} · {seconds} s")
    return result


def text_of(result: dict | None) -> str:
    return " ".join(p["text"] for p in result["reply"].get("combinedPhrases", [])) if result else ""


def words(text: str) -> list[str]:
    """Lowercase, no accents or punctuation: the rules we score by. Decide these before you look at the scores."""
    text = unicodedata.normalize("NFKD", text.lower())
    return re.findall(r"[a-z0-9$]+", "".join(c for c in text if not unicodedata.combining(c)))


def word_error_rate(expected: str, heard: str) -> tuple[float, list[str]]:
    """How many words we'd have to change to fix the transcript, as a share of the script. 0 is perfect."""
    want, got = words(expected), words(heard)
    changes, notes = 0, []
    for op, a1, a2, b1, b2 in difflib.SequenceMatcher(a=want, b=got, autojunk=False).get_opcodes():
        if op != "equal":
            changes += max(a2 - a1, b2 - b1)
            notes.append(f"{op}: {' '.join(want[a1:a2]) or '∅'} → {' '.join(got[b1:b2]) or '∅'}")
    return changes / max(len(want), 1), notes

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.5 Did it hear the ad right?

<img src="./assets/asks/03-05.svg" alt="Your customer asks: Can we trust a transcript enough to act on it?" width="900" />
<br/>

Let's start with the English radio spot from 02. We know the script exactly, so the score is honest.

> 🔮 **Guess first:** how many words will be different? Watch *$250* and *X4*.

In [4]:
key = scripts()
radio = transcribe("05-radio-en", "05-radio-en")
if radio and "05-radio-en" in key:
    rate, notes = word_error_rate(key["05-radio-en"]["text"], text_of(radio))
    print(f"\n📝 Heard: {text_of(radio)}\n")
    print(f"📏 Word error rate: {rate:.2f}  ({len(notes)} differences)")
    for note in notes:
        print("   ", note)

👂 05-radio-en · MAI-Transcribe-2 · 0.73 s

📝 Heard: Rain in the forecast? Bring it! The Trailmaster X4 from Outdoor Living sleeps four, stands up on its own, and comes with its own rainfly. Home, anywhere, for $250. Find it at Contoso Outdoors.

📏 Word error rate: 0.06  (1 differences)
    replace: outdoorliving → outdoor living


👀 **What to look for:**

- [ ] Is the meaning right, even if a few words differ?
- [ ] Look at each difference. Is it a real mistake, or a **formatting choice**, like *$250* versus *250 dollars*?
- [ ] Is the brand spelled right: *TrailMaster*, *OutdoorLiving*?

⚠️ **The honest limit:** word error rate counts *$250* versus *two hundred fifty dollars* as a mistake, even though both are right. Decide your scoring rules first, write them down, and keep them fixed, or the score will move for reasons that have nothing to do with the model.

🧪 **Try it yourself:** change `words()` so *$250* and *250 dollars* count as the same. Did the score improve for a good reason?

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.6 Spell our names our way

<img src="./assets/asks/03-06.svg" alt="Your customer asks: Our product names are unusual. Will it spell them right?" width="900" />
<br/>

*TrailMaster*, *OutdoorLiving*, *rainfly*, *guy lines*: a general model might write them three different ways. **Keyword hints** tell it which spellings to expect.

Same audio, the setup guide from 02, transcribed twice: without hints, then with them.

> 🔮 **Guess first:** will the hints change anything on clean, clearly spoken audio?

In [5]:
NAMES = ["TrailMaster X4", "OutdoorLiving", "Contoso Outdoors", "rainfly", "guy lines"]

plain = transcribe("06-guide-plain", "09-setup-guide")
hinted = transcribe("06-guide-hints", "09-setup-guide", hints=NAMES)
expected = key.get("09-setup-guide", {}).get("text", "")
for label, result in (("without hints", plain), ("with hints", hinted)):
    if result and expected:
        rate, _ = word_error_rate(expected, text_of(result))
        found = {name: text_of(result).count(name) for name in NAMES}
        print(f"{label:14} WER {rate:.2f} · exact spellings found: {found}")

👂 06-guide-plain · MAI-Transcribe-2 · 0.97 s
👂 06-guide-hints · MAI-Transcribe-2 · 0.75 s
without hints  WER 0.00 · exact spellings found: {'TrailMaster X4': 0, 'OutdoorLiving': 0, 'Contoso Outdoors': 0, 'rainfly': 5, 'guy lines': 4}
with hints     WER 0.00 · exact spellings found: {'TrailMaster X4': 0, 'OutdoorLiving': 0, 'Contoso Outdoors': 0, 'rainfly': 5, 'guy lines': 4}


👀 **What to look for:**

- [ ] Do the product names appear with **exactly** our spelling?
- [ ] Did the hints change the score, or only the spelling of a few words?
- [ ] Did a hint cause a *new* mistake, like hearing a product name that wasn't said?

⚠️ **The honest limit:** clean, synthetic audio is easy. Real calls have accents, noise and people talking over each other. Hints matter more there, and so does a test set of **real** recordings.

🧪 **Try it yourself:** record yourself reading two sentences with *TrailMaster X4* in them, save it to `output/02/my-voice.mp3`, and transcribe it with and without hints.

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.7 Which language is this?

<img src="./assets/asks/03-07.svg" alt="Your customer asks: Our ads run in three languages. Does it need to be told which?" width="900" />
<br/>

We don't tell the model which language to expect. It works it out from the audio, for each clip.

> 🔮 **Guess first:** will the Spanish and French spots score as well as the English one?

In [6]:
for code_ in ("es", "fr"):
    name = f"07-radio-{code_}"
    result = transcribe(name, name)
    if result and name in key:
        rate, notes = word_error_rate(key[name]["text"], text_of(result))
        locale = {p.get("locale") for p in result["reply"].get("phrases", []) if p.get("locale")}
        print(f"\n{code_}: detected {sorted(locale) or 'not reported'} · WER {rate:.2f}")
        print(f"   heard: {text_of(result)}")
        for note in notes[:5]:
            print("   ", note)

👂 07-radio-es · MAI-Transcribe-2 · 0.28 s

es: detected ['es'] · WER 0.08
   heard: ¿Lluvia en el pronóstico? Que venga. La Trailmaster X4 de Outdoor Living es para 4 personas, se sostiene sola y trae su propio doble techo. Tu casa en cualquier lugar por 250 dólares. Encuéntrala en Contoso Outdoors.
    replace: outdoorliving → outdoor living
    replace: cuatro → 4
👂 07-radio-fr · MAI-Transcribe-2 · 0.28 s

fr: detected ['fr'] · WER 0.11
   heard: De la pluie annoncée ? Qu'elle vienne ! La Trailmaster X4 d'Outdoor Living accueille 4 personnes, tient debout toute seule et inclut son double toit. Votre maison ? Partout, pour 250 dollars. En vente chez Contoso Outdoors.
Oh.
    replace: outdoorliving → outdoor living
    replace: quatre → 4
    insert: ∅ → oh


👀 **What to look for:**

- [ ] Did it detect the right language for each clip?
- [ ] Are the differences real mistakes, or spelling choices like *250* versus *doscientos cincuenta*?
- [ ] Is the brand name the same in all three languages?

⚠️ **The honest limit:** our scoring rules were written for English. In other languages, numbers and accents need their own rules. A native speaker should review both the transcript and the rules.

🧪 **Try it yourself:** add keyword hints in Spanish, like *doble techo*. Does the Spanish score change?

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.8 Who said what?

<img src="./assets/asks/03-08.svg" alt="Your customer asks: Two people, one recording. Can it tell who said what?" width="900" />
<br/>

The support call from 02 has two people: a customer and an agent. With **diarization**, the model labels each phrase with a speaker number.

> 🔮 **Guess first:** will it split the four turns between two speakers correctly?

In [11]:
call = transcribe("08-call-speakers", "10-support-call", speakers=2)
if call:
    for phrase in call["reply"].get("phrases", []):
        start = phrase.get("offsetMilliseconds", 0) / 1000
        print(f"[{start:5.1f} s] speaker {phrase.get('speaker', '?')}: {phrase['text']}")
    turns = key.get("10-support-call", {}).get("turns", [])
    if turns:
        print("\n📋 The script said:")
        for who, line in turns:
            print(f"   {who}: {line}")

👂 08-call-speakers · MAI-Transcribe-2 · 1.5 s
[  0.2 s] speaker 0: Hi. Can I use the Trailmaster X4 in winter?
[  3.8 s] speaker 1: Good question. The Trailmaster X4 is designed for three-season use and may not be suitable for extreme winter conditions with heavy snow and freezing temperatures.
[ 14.2 s] speaker 0: Okay.
[ 15.0 s] speaker 0: And is there a warranty?
[ 16.8 s] speaker 1: Yes, the Trailmaster X4 comes with a two-year limited warranty against manufacturing defects.

📋 The script said:
   customer: Hi. Can I use the TrailMaster X4 in winter?
   agent: Good question. The TrailMaster X4 is designed for 3-season use, and may not be suitable for extreme winter conditions with heavy snow and freezing temperatures.
   customer: Okay. And is there a warranty?
   agent: Yes. The TrailMaster X4 comes with a 2-year limited warranty against manufacturing defects.


👀 **What to look for:**

- [ ] Are there exactly **two** speakers?
- [ ] Does every agent line land on the same speaker number?
- [ ] Are the facts intact: *3-season*, *2-year limited warranty*?

⚠️ **The honest limit:** the model numbers speakers; it doesn't know who is the agent. Something else has to map *speaker 1* to *agent*, like who spoke first or a known greeting.

🧪 **Try it yourself:** set `speakers=3` on the same clip. Does it invent a third person?

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.9 Search every call

<img src="./assets/asks/03-09.svg" alt="Your customer asks: We have hours of calls. Can we find every mention of the rainfly?" width="900" />
<br/>

Once speech is text, it's searchable. Each phrase comes with a time, so we can jump straight to the moment someone said it.

> 🔮 **Guess first:** in which clips will *rainfly* turn up?

In [12]:
TERM = "rainfly"

transcripts = {path.stem: json.loads(path.read_text()) for folder in (EXAMPLES, OUTPUT) for path in sorted(folder.glob("*.json"))}
hits = 0
for name, result in transcripts.items():
    for phrase in result["reply"].get("phrases", []):
        if TERM.lower() in phrase["text"].lower():
            hits += 1
            print(f"🔎 {name} at {phrase.get('offsetMilliseconds', 0) / 1000:.1f} s: …{phrase['text']}…")
print(f"\n{hits} mentions of '{TERM}' across {len(transcripts)} transcripts")

🔎 05-radio-en at 0.0 s: …Rain in the forecast? Bring it! The Trailmaster X4 from Outdoor Living sleeps four, stands up on its own, and comes with its own rainfly. Home, anywhere, for $250. Find it at Contoso Outdoors.…
🔎 06-guide-hints at 0.0 s: …Step 1: Selecting a suitable location. Find a level and clear area for pitching the tent. Remove any sharp objects or debris that could damage the tent floor. Step 2: Unpacking and organizing components. Lay out all the tent components on the ground. Familiarize yourself with each part, including the tent body, poles, rainfly, stakes, and guy lines. Step 3: Assembling the tent poles. Connect the tent poles according to their designated color codes or numbering. Slide the connected poles through the pole sleeves or attach them to the tent body clips. Step 4: Setting up the tent body. Begin at one end and raise the tent body by pushing up the poles. Ensure that the tent body is evenly stretched and centered. Secure the tent body to the ground us

👀 **What to look for:**

- [ ] Did it find *rainfly* in the radio spot and in the setup guide?
- [ ] Does each hit come with a time you could jump to?
- [ ] Did the French and Spanish clips match? They say *double toit* and *doble techo*.

⚠️ **The honest limit:** plain text search only finds the exact word. To search by meaning across languages, you'd index the transcripts and search them with an embedding model or AI Search. That's a step beyond this notebook.

🧪 **Try it yourself:** search for `warranty`, then for `winter`. Which clip answers each?

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.10 Live captions

<img src="./assets/asks/03-10.svg" alt="Your customer asks: Our launch stream needs live captions. Can it keep up?" width="900" />
<br/>

Everything so far was **batch**: send a finished file, get a transcript back. Live captions need **streaming**: words arrive while the person is still talking, first as quick guesses, then as final text.

MAI-Transcribe-2 has a streaming version, **MAI-Transcribe-2-Streaming**, in preview. It needs a live audio connection rather than a file upload, so we don't run it here.

| | Batch (this notebook) | Streaming |
| --- | --- | --- |
| Input | A finished file | Live audio, as it happens |
| Output | One transcript | Quick partial results, then final text |
| Good for | Calls, ads, archives, search | Live captions, voice agents |

🧪 **Go further:** see the [MAI-Transcribe docs](https://learn.microsoft.com/azure/ai-services/speech-service/mai-transcribe) for streaming, and try captioning the launch stream at home.

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.11 Teach it to a customer

### What it did, and where to show it

| Job | Why a customer cares | Show them |
| --- | --- | --- |
| Checked transcripts | Decisions based on what was really said | 3.5 |
| Keyword hints | Product names spelled their way | 3.6 |
| Automatic language detection | One pipeline for every market | 3.7 |
| Speaker labels | Calls you can review by who spoke | 3.8 |
| Searchable calls | Find every mention, with a time | 3.9 |

### Honest answers to common questions

| They ask… | You can say… |
| --- | --- |
| How accurate is it? | Measure it on **your** audio, with your own answer key, like 3.5. Don't quote someone else's benchmark. |
| Will it get our product names? | Use keyword hints, then check on real calls. |
| Which languages? | Many, with automatic detection. Check the docs for your markets. |
| Can it caption live? | Yes, with the streaming version, which is in preview. |

### When something goes wrong

| You see | It usually means | Try |
| --- | --- | --- |
| `401` | Not signed in | `az login` |
| `403` | No access | Foundry User role on the Foundry resource |
| `404` | Wrong region or model name | Check `MODEL_TRANSCRIBE` and your resource's region |
| `no audio for …` | Notebook 02 hasn't made that clip | Run 02 live first |

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.12 Check yourself

1. Your transcript says *250 dollars* and the script says *$250*. Is that a mistake?
2. Why is synthetic audio a weak test set for a call center?
3. When would you pick the streaming version?

<details>
<summary>Show answers</summary>

1. Not a real one, but your score counts it unless your rules say otherwise. Write the rules down before you score.
2. It's clean: no accents, noise or interruptions. Real recordings are harder, so test on them.
3. When someone needs the words while the person is still talking: live captions or a voice agent.

</details>

### 🤖 Ask your agent

- *"Write normalization rules for scoring transcripts that treat numbers, currency and contractions fairly."*
- *"Sketch a pipeline that transcribes support calls nightly and flags any that mention a product defect."*

<img src="./assets/divider.svg" alt="" width="100%" height="6" />

## 3.13 Cost and cleanup

This notebook doesn't create any Azure resources. You pay per hour of audio transcribed. When you're done:

- **Keep `output/03/`** if you want to show your transcripts in replay mode. Git ignores the folder.
- **Delete `output/03/`** to clear them.

➡️ **Next:** [04 · Refresh the storefront](./04-refresh-the-storefront.ipynb). The launch kit needs a home on the web.